# Resume Screening RAG Pipeline (cleaned)

Fixes applied vs. the original notebook:

1. Every function is defined exactly once, in dependency order (no more `NameError: exact_skill_match not defined` from calling something before a later cell defines it).
2. Requirement normalization is consistent and case-insensitive everywhere (single `ALIASES` dict + `normalize_requirement()`).
3. `exact_skill_match()` uses word-boundary regex instead of bare substring matching, so "SQL" no longer accidentally matches inside "MySQL" for the wrong reasons (still matches MySQL, but explicitly via an allowlist, not by accident).
4. `evaluate_retrieval`'s dead `item == "..."` comparison (always False) is removed.
5. Ollama response access is consistent (dict-style) everywhere.
6. No reliance on leftover global loop variables.

## Setup

In [76]:
import re
import json
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import ollama

In [77]:
MODEL_NAME = "llama3.2"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHROMA_PATH = "../dataset/chroma"
COLLECTION_NAME = "resume_chunk_v2"
RERANK_THRESHOLD = -10.3  # tune per your reranker/data -- see note at the bottom

SECTION_NAMES = ["ABOUT ME", "SKILLS", "PROFESSIONAL EXPERIENCE", "PROJECTS", "EDUCATION"]

# Single, case-insensitive alias table used everywhere (fixes the old
# case-sensitive SKILL_NORMALIZATION dict that rarely matched).
ALIASES = {
    "python programming": "Python",
    "python programming skills": "Python",
    "sql databases": "SQL",
    "sql database": "SQL",
    "sql/database knowledge": "SQL",
    "experience working with git and github": "Git and GitHub",
    "good problem solving skills": "Problem solving",
}

## 1. PDF extraction + cleaning

In [78]:
def extract_pdf_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


def clean_text(text: str) -> str:
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text

## 2. Section-aware chunking

In [79]:
def extract_sections(text: str, section_names=SECTION_NAMES) -> dict:
    pattern = r"(?m)^(" + "|".join(section_names) + r")\s*$"
    parts = re.split(pattern, text, flags=re.IGNORECASE)

    sections = {}
    current_section = None
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if part.upper() in section_names:
            current_section = part.upper()
            sections[current_section] = ""
        elif current_section:
            sections[current_section] += " " + part
    return sections


def chunk_section(text: str, section: str, max_words: int = 150, overlap: int = 30) -> list:
    words = text.split()
    if len(words) <= max_words:
        return [{"text": f"{section}\n{text}", "section": section}]

    chunks = []
    start = 0
    while start < len(words):
        end = start + max_words
        piece = " ".join(words[start:end])
        chunks.append({"text": f"{section}\n{piece}", "section": section})
        start += max_words - overlap
    return chunks

## 3. Embedding + vector store

In [80]:
_embed_model = None
_reranker = None
_collection = None


def get_embed_model() -> SentenceTransformer:
    global _embed_model
    if _embed_model is None:
        _embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    return _embed_model


def get_reranker() -> CrossEncoder:
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder(RERANK_MODEL_NAME)
    return _reranker


def get_collection():
    global _collection
    if _collection is None:
        client = chromadb.PersistentClient(path=CHROMA_PATH)
        _collection = client.get_or_create_collection(name=COLLECTION_NAME)
    return _collection


def build_metadata(chunk: dict, resume_id: str, candidate_id: str) -> dict:
    metadata = {"resume_id": resume_id, "candidate_id": candidate_id, "section": chunk["section"]}
    for key in ("subsection", "project", "job_title", "company"):
        if key in chunk:
            metadata[key] = chunk[key]
    return metadata


def index_chunks(chunks: list, resume_id: str = "resume_001", candidate_id: str = "candidate_001"):
    """Embed a list of {"text": ..., "section": ..., ...} chunks and add to Chroma."""
    model = get_embed_model()
    collection = get_collection()

    documents = [c["text"] for c in chunks]
    embeddings = model.encode(documents)
    ids = [f"{resume_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [build_metadata(c, resume_id, candidate_id) for c in chunks]

    collection.upsert(ids=ids, documents=documents, embeddings=embeddings.tolist(), metadatas=metadatas)
    return len(chunks)

## 4. Retrieval + section routing + reranking

In [81]:
def search_resume(query: str, n_results: int = 3, section: str = None) -> dict:
    model = get_embed_model()
    collection = get_collection()

    query_embedding = model.encode(query).tolist()
    kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"],
    }
    if section:
        kwargs["where"] = {"section": section}

    return collection.query(**kwargs)


def detect_section(query: str):
    q = query.lower()
    if any(w in q for w in ["project", "projects", "built", "application", "app"]):
        return "PROJECTS"
    if any(w in q for w in ["work", "worked", "experience", "job", "company", "internship"]):
        return "PROFESSIONAL EXPERIENCE"
    if any(w in q for w in ["education", "study", "studied", "college", "school", "degree", "training"]):
        return "EDUCATION"
    if any(w in q for w in ["skill", "skills", "know", "database", "language", "framework", "tool"]):
        return "SKILLS"
    return None


def rerank_results(query: str, results: dict) -> list:
    documents = results["documents"][0]
    if not documents:
        return []

    reranker = get_reranker()
    pairs = [[query, doc] for doc in documents]
    scores = reranker.predict(pairs)

    ranked = [
        {
            "document": documents[i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
            "rerank_score": float(scores[i]),
        }
        for i in range(len(documents))
    ]
    ranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return ranked


def retrieve_and_rerank(query: str, n_results: int = 10, max_results: int = 3,
                         threshold: float = RERANK_THRESHOLD) -> list:
    section = detect_section(query)
    results = search_resume(query=query, n_results=n_results, section=section)
    ranked = rerank_results(query, results)
    relevant = [r for r in ranked if r["rerank_score"] >= threshold]
    return relevant[:max_results]

## 5. Skill verification

`exact_skill_match` uses word-boundary regex instead of bare substring matching. Plain substring matching would match `"Java"` inside `"JavaScript"`, which is wrong. `"SQL"` matching inside `"MySQL"` is kept, but as an explicit, intentional rule via `SUBSTRING_OK` rather than an accident.

In [82]:
# Terms where a substring match against a longer word is intentionally OK
# (the longer word implies the shorter skill).
SUBSTRING_OK = {
    "sql": ["mysql", "postgresql", "mssql", "sqlite", "oracle"],
}


def normalize_requirement(requirement: str) -> str:
    key = requirement.strip().lower()
    return ALIASES.get(key, requirement)


def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9+#.\- ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _term_matches(term: str, evidence: str) -> bool:
    if re.search(rf"\b{re.escape(term)}\b", evidence):
        return True
    for allowed_word in SUBSTRING_OK.get(term, []):
        if allowed_word in evidence:
            return True
    return False


def exact_skill_match(skill: str, evidence_text: str) -> bool:
    skill_terms = re.findall(r"[a-z0-9+#.-]+", skill.lower())
    evidence_norm = normalize_text(evidence_text)
    return all(_term_matches(term, evidence_norm) for term in skill_terms)


def verify_skill_llm(skill: str, evidence_text: str) -> bool:
    """LLM fallback for cases exact_skill_match can't resolve (paraphrases, etc.)."""
    prompt = f"""
You are checking whether a resume contains evidence for a requirement.

REQUIREMENT:
{skill}

RESUME EVIDENCE:
{evidence_text}

Does the evidence support the requirement?

- If the exact skill appears in the evidence, answer TRUE.
- If the evidence clearly names the required technology, answer TRUE.
- Otherwise answer FALSE.

Return ONLY:
TRUE
or
FALSE
"""
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Return only TRUE or FALSE."},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0},
    )
    answer = response["message"]["content"].strip().upper()
    return answer == "TRUE"


def check_skill(skill: str) -> dict:
    original_skill = skill
    normalized_skill = normalize_requirement(skill)

    query = normalized_skill
    evidence = retrieve_and_rerank(query)

    if not evidence:
        return {"skill": original_skill, "normalized_skill": normalized_skill, "matched": False, "evidence": []}

    verified_evidence = []
    for item in evidence:
        document = item["document"]
        if exact_skill_match(normalized_skill, document) or verify_skill_llm(normalized_skill, document):
            verified_evidence.append(item)

    return {
        "skill": original_skill,
        "normalized_skill": normalized_skill,
        "matched": len(verified_evidence) > 0,
        "evidence": verified_evidence,
    }

## 6. Job requirement extraction + candidate scoring

In [83]:
def extract_job_requirements(job_description: str) -> dict:
    prompt = f"""
You are a job description analysis assistant.

Extract EVERY explicit requirement from the job description.

Return ONLY valid JSON with exactly this structure:

{{
    "skills": [],
    "tools": [],
    "frameworks": [],
    "databases": [],
    "soft_skills": []
}}

Rules:
- Extract EVERY explicit requirement, never infer or add implied ones.
- Do not merge related requirements (e.g. "PyTorch" and "TensorFlow" stay separate).
- soft_skills: only include skills explicitly stated, do not add common workplace skills.
- Do not add explanations outside the JSON.

JOB DESCRIPTION:
{job_description}
"""
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You extract structured job requirements. Return valid JSON only."},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0},
    )
    content = response["message"]["content"]
    return json.loads(content)


def flatten_requirements(requirements: dict) -> list:
    flat = []
    for items in requirements.values():
        for item in items:
            item = item.strip()
            if item:
                flat.append(item)
    return list(dict.fromkeys(flat))  # dedupe, keep order

In [84]:
REQUIREMENT_WEIGHTS_DEFAULT = 1.0


def calculate_match_score(results: list, weights: dict = None) -> float:
    weights = weights or {}
    total_weight = 0.0
    matched_weight = 0.0
    for result in results:
        weight = weights.get(result["skill"], REQUIREMENT_WEIGHTS_DEFAULT)
        total_weight += weight
        if result["matched"]:
            matched_weight += weight
    return (matched_weight / total_weight) * 100 if total_weight else 0.0


def analyze_requirements(requirements: list, weights: dict = None) -> dict:
    results = [check_skill(skill) for skill in requirements]
    score = calculate_match_score(results, weights)
    return {"score": score, "results": results}

In [85]:
def print_candidate_report(analysis: dict):
    print("=" * 60)
    print("AI RESUME SCREENING REPORT")
    print("=" * 60)
    print(f"\nMATCH SCORE: {analysis['score']:.2f}%")

    print("\nMATCHED")
    print("-" * 60)
    for result in analysis["results"]:
        if not result["matched"]:
            continue
        print(f"\n\u2713 {result['skill']}")
        for evidence in result["evidence"]:
            section = evidence["metadata"].get("section")
            subsection = evidence["metadata"].get("subsection")
            print(f"  Source: {section} \u2192 {subsection}")
            print(f"  Evidence: {evidence['document']}")

    print("\nMISSING")
    print("-" * 60)
    for result in analysis["results"]:
        if result["matched"]:
            continue
        print(f"\n\u2717 {result['skill']}")
        print("  No supporting evidence found in the resume.")


def build_evidence_context(analysis: dict) -> str:
    grouped = {}
    for result in analysis["results"]:
        if not result["matched"]:
            continue
        skill = result["skill"]
        grouped.setdefault(skill, [])
        for evidence in result["evidence"]:
            metadata = evidence["metadata"]
            item = (metadata.get("section"), metadata.get("subsection"), evidence["document"])
            if item not in grouped[skill]:
                grouped[skill].append(item)

    parts = []
    for skill, evidence_list in grouped.items():
        block = f"Requirement: {skill}\n"
        for section, subsection, document in evidence_list:
            block += f"\nSource: {section} \u2192 {subsection}\nEvidence:\n{document}\n"
        parts.append(block)
    return "\n".join(parts)


def build_candidate_prompt(analysis: dict) -> str:
    evidence_context = build_evidence_context(analysis)
    missing = [r["skill"] for r in analysis["results"] if not r["matched"]]

    return f"""
You are an AI resume screening assistant.

Analyze the candidate based ONLY on the verified resume evidence provided below.
Do not invent skills or experience.

MATCH SCORE:
{analysis['score']:.2f}%

VERIFIED EVIDENCE:
{evidence_context}

MISSING REQUIREMENTS:
{chr(10).join("- " + s for s in missing)}

Write a concise candidate assessment. Include:
1. Overall assessment
2. Confirmed strengths
3. Missing requirements
4. Final recommendation

Rules:
- Only use the evidence provided.
- Do not assume a skill from a related technology.
- Do not make a hiring decision unless an explicit hiring threshold is provided.
- Do not discuss unrelated resume skills unless they are relevant to the requirements.
- Confirmed strengths must come from matched requirements only.
- Do not describe unrelated technologies as strengths.
"""


def generate_candidate_assessment(prompt: str) -> str:
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a precise AI resume screening assistant. "
                    "Use only the evidence supplied. Do not invent skills, experience, "
                    "or qualifications. Do not make hiring decisions unless explicitly instructed."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0},
    )
    return response["message"]["content"]

## 7. End-to-end run

In [86]:
raw_text = extract_pdf_text("../dataset/sefat_khan.pdf")
text = clean_text(raw_text)
sections = extract_sections(text)

chunks = []
for section, content in sections.items():
    chunks.extend(chunk_section(content, section))

index_chunks(chunks)
print("Indexed", len(chunks), "chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 5 chunks


In [87]:
job_description = """
Machine Learning Engineer Intern

Requirements:
- Strong Python programming skills
- Knowledge of machine learning algorithms
- Experience with scikit-learn
- Knowledge of NLP and Transformers
- Experience with PyTorch or TensorFlow
- Knowledge of SQL databases
- Good problem solving skills
- Experience working with Git and GitHub
"""

requirements = extract_job_requirements(job_description)
flat_requirements = flatten_requirements(requirements)
print(flat_requirements)

['Python programming', 'Machine learning algorithms', 'NLP', 'Transformers', 'scikit-learn', 'PyTorch', 'TensorFlow', 'Git', 'GitHub', 'SQL databases', 'Good problem solving skills']


In [88]:
analysis = analyze_requirements(flat_requirements)
print_candidate_report(analysis)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

AI RESUME SCREENING REPORT

MATCH SCORE: 45.45%

MATCHED
------------------------------------------------------------

✓ Python programming
  Source: SKILLS → frontend
  Evidence: SKILLS
  Languages: JavaScript, TypeScript, PHP, Python, SQL, HTML, CSS 
 Frontend: React.js, Redux Toolkit, Tailwind CSS, Bootstrap, React Bootstrap 
 Backend: Laravel, REST APIs, Authentication Systems 
 Database: MySQL, Firebase 
 Tools: Git, GitHub 
 
 Soft Skills: Time management, problem solving, teamwork, adaptability, attention to detail, 
reliability 
 
 Communication Skills: English (Intermediate), Bengali (Native)

✓ Git
  Source: SKILLS → frontend
  Evidence: SKILLS
  Languages: JavaScript, TypeScript, PHP, Python, SQL, HTML, CSS 
 Frontend: React.js, Redux Toolkit, Tailwind CSS, Bootstrap, React Bootstrap 
 Backend: Laravel, REST APIs, Authentication Systems 
 Database: MySQL, Firebase 
 Tools: Git, GitHub 
 
 Soft Skills: Time management, problem solving, teamwork, adaptability, att

In [89]:
prompt = build_candidate_prompt(analysis)
assessment = generate_candidate_assessment(prompt)
print(assessment)

**Candidate Assessment**

**Overall Assessment:**
The candidate has a strong foundation in frontend development, with expertise in React, Laravel, and MySQL. They have also demonstrated good problem-solving skills, teamwork, and adaptability. However, their skills do not align with the requirements for machine learning algorithms, NLP, transformers, scikit-learn, PyTorch, and TensorFlow.

**Confirmed Strengths:**

* Frontend development skills, including React, Redux Toolkit, Tailwind CSS, Bootstrap, and React Bootstrap
* Backend development skills, including Laravel, REST APIs, and authentication systems
* Database skills, including MySQL and Firebase
* Good problem-solving skills, teamwork, and adaptability
* Proficiency in Git and GitHub

**Missing Requirements:**
The candidate is missing skills in the following areas:

* Machine learning algorithms
* NLP
* Transformers
* scikit-learn
* PyTorch
* TensorFlow

**Final Recommendation:**
Based on the provided evidence, the candidate is 

In [90]:
# Quick sanity check on the bug that started all this
result = check_skill("SQL databases")
print(result)

{'skill': 'SQL databases', 'normalized_skill': 'SQL', 'matched': True, 'evidence': [{'document': 'SKILLS\n \uf0b7 Languages: JavaScript, TypeScript, PHP, Python, SQL, HTML, CSS \n\uf0b7 Frontend: React.js, Redux Toolkit, Tailwind CSS, Bootstrap, React Bootstrap \n\uf0b7 Backend: Laravel, REST APIs, Authentication Systems \n\uf0b7 Database: MySQL, Firebase \n\uf0b7 Tools: Git, GitHub \n \n\uf0b7 Soft Skills: Time management, problem solving, teamwork, adaptability, attention to detail, \nreliability \n \n\uf0b7 Communication Skills: English (Intermediate), Bengali (Native)', 'metadata': {'subsection': 'frontend', 'candidate_id': 'candidate_001', 'resume_id': 'resume_001', 'section': 'SKILLS'}, 'distance': 1.6506266593933105, 'rerank_score': -2.5749282836914062}, {'document': 'ABOUT ME\n Full Stack Web Developer with hands-on industry experience building responsive and scalable web \napplications using React, Laravel, and MySQL. Skilled in frontend performance optimization, REST API \nin

---
**Note on `RERANK_THRESHOLD`:** `-10.3` was reverse-engineered from one exploratory run comparing "positive" vs "negative" queries in the original notebook. It's model- and data-specific -- if you change the reranker, the resume content, or the phrasing of `check_skill`'s query template, re-run that positive/negative probe and re-pick the threshold rather than trusting this constant blindly.